# LASSA SEROPREVALENCE MACHINE LEARNING
## SECTION 3: PREPROCESSING, FEATURE ENGINEERING & MODEL-READY DATA

**Goal:** Convert raw dataset into clean, encoded, scaled, train/test split arrays ready for model training.

---

### Why Section 3 Matters
- ML models require **numerical** inputs → categorical features must be encoded
- Models perform better with consistent numerical scales → scaling
- We must prevent data leakage → split before fitting transformations
- Your dataset has **extreme class imbalance (3 positives)** → careful splitting + imbalance strategy

---

### Outputs Produced
- `df_clean`: cleaned dataset
- `X`, `y`: features/target
- `X_train`, `X_test`, `y_train`, `y_test`
- `preprocess`: sklearn ColumnTransformer pipeline
- `X_train_processed`, `X_test_processed`
- Baseline model results + sanity metrics

---

### Critical Warning
You have **only 3 positive samples**. That makes train/test evaluation unstable.
We add a safety split routine to guarantee the test set has at least 1 positive (if possible).


---

## CELL 3.0: LOAD DATA (LOCAL ONLY)

### What We Do
- Load the CSV from your local folder.
- Create `df_raw` and `df`.

### Why
Each notebook runs in its own kernel session, so Section 3 must load the data again.


In [1]:
import os
import pandas as pd

print("\n" + "="*100)
print("CELL 3.0: LOAD DATA (LOCAL ONLY)")
print("="*100)

# IMPORTANT: adjust this path only if your setup changes
data_path = "data/embeddings/data/LASV_Master_Data!.csv"

print("Current working directory:", os.getcwd())
print("Trying to load:", data_path)

if not os.path.exists(data_path):
    print("\n✗ File not found:", data_path)
    print("\nFiles in current folder:", os.listdir("."))
    if os.path.exists("data"):
        print("\nFiles in data/:", os.listdir("data"))
    raise FileNotFoundError(data_path)

df_raw = pd.read_csv(data_path)
df = df_raw.copy()

print("\n✓ Loaded successfully")
print("Shape:", df.shape)
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicates:", int(df.duplicated().sum()))
print("="*100)



CELL 3.0: LOAD DATA (LOCAL ONLY)
Current working directory: /Users/user
Trying to load: data/embeddings/data/LASV_Master_Data!.csv

✓ Loaded successfully
Shape: (250, 46)
Missing values: 0
Duplicates: 0


---

## CELL 3.1: BASIC CLEANING (STRING NORMALIZATION + DROP COLUMNS)

### What We Do
1. Strip whitespace in all string columns (fixes issues like `"Female "` vs `"Female"`).
2. Drop `Full_Name` (requested).
3. Drop `Patient_ID` (recommended: it’s an identifier, not a predictive feature).

### Why This Matters
- Prevents duplicate categories due to trailing spaces
- Prevents ID leakage and improves generalization


In [2]:
import numpy as np

print("\n" + "="*100)
print("CELL 3.1: BASIC CLEANING")
print("="*100)

df_clean = df.copy()

# 1) Strip whitespace for ALL object columns
obj_cols = df_clean.select_dtypes(include=["object"]).columns
for c in obj_cols:
    df_clean[c] = df_clean[c].astype(str).str.strip()

# 2) Drop columns
cols_to_drop = []
if "Full_Name" in df_clean.columns:
    cols_to_drop.append("Full_Name")
if "Patient_ID" in df_clean.columns:
    cols_to_drop.append("Patient_ID")

df_clean = df_clean.drop(columns=cols_to_drop, errors="ignore")

print("✓ Whitespace stripped from object columns")
print("✓ Dropped columns:", cols_to_drop)
print("\nNew shape:", df_clean.shape)

# Quick sanity check for Gender
if "Gender" in df_clean.columns:
    print("\nGender value_counts after strip():")
    print(df_clean["Gender"].value_counts())

print("="*100)



CELL 3.1: BASIC CLEANING
✓ Whitespace stripped from object columns
✓ Dropped columns: ['Full_Name', 'Patient_ID']

New shape: (250, 44)

Gender value_counts after strip():
Gender
Female    165
Male       85
Name: count, dtype: int64


---

## CELL 3.2: TARGET MAPPING (TO 0/1)

### What We Do
- Convert `lab_results.PCR_Results` into numeric labels:
  - `No Kb (Negative)` → 0
  - `320Kb (Positive)` → 1

### Why
Most ML algorithms require numeric target labels.


In [3]:
print("\n" + "="*100)
print("CELL 3.2: TARGET MAPPING")
print("="*100)

target_col = "lab_results.PCR_Results"

if target_col not in df_clean.columns:
    raise KeyError(f"Target column not found: {target_col}")

print("Original target distribution:")
print(df_clean[target_col].value_counts(dropna=False))

mapping = {
    "No Kb (Negative)": 0,
    "320Kb (Positive)": 1
}

y = df_clean[target_col].map(mapping)

# Validate mapping
unmapped = df_clean.loc[y.isna(), target_col].unique().tolist()
if len(unmapped) > 0:
    raise ValueError(f"Unmapped target values found: {unmapped}")

y = y.astype(int)

# Create X
X = df_clean.drop(columns=[target_col])

print("\n✓ Target mapped successfully")
print("y distribution (0=Negative, 1=Positive):")
print(pd.Series(y).value_counts())
print("\nX shape:", X.shape)

print("="*100)



CELL 3.2: TARGET MAPPING
Original target distribution:
lab_results.PCR_Results
No Kb (Negative)    247
320Kb (Positive)      3
Name: count, dtype: int64

✓ Target mapped successfully
y distribution (0=Negative, 1=Positive):
lab_results.PCR_Results
0    247
1      3
Name: count, dtype: int64

X shape: (250, 43)


---

## CELL 3.3: TRAIN/TEST SPLIT (STRATIFIED + SAFETY CHECK)

### Why Special Split?
You have only **3 positives**. A random split can produce **0 positives in the test set**, which breaks evaluation.

### What We Do
- Use stratified split.
- Retry up to 200 times until test set contains at least 1 positive.


In [4]:
from sklearn.model_selection import train_test_split

print("\n" + "="*100)
print("CELL 3.3: TRAIN/TEST SPLIT")
print("="*100)

TEST_SIZE = 0.2
RANDOM_SEED = 42

def safe_stratified_split(X, y, test_size=0.2, base_seed=42, max_tries=200, min_pos_test=1):
    y = np.array(y)
    for i in range(max_tries):
        seed = base_seed + i
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y,
            test_size=test_size,
            random_state=seed,
            stratify=y
        )
        if (y_te == 1).sum() >= min_pos_test:
            return X_tr, X_te, y_tr, y_te, seed
    return X_tr, X_te, y_tr, y_te, seed

X_train, X_test, y_train, y_test, used_seed = safe_stratified_split(
    X, y,
    test_size=TEST_SIZE,
    base_seed=RANDOM_SEED,
    max_tries=200,
    min_pos_test=1
)

print(f"✓ Split completed using random_state={used_seed}")
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("y_train positives:", int((y_train == 1).sum()), "/", len(y_train))
print("y_test positives:", int((y_test == 1).sum()), "/", len(y_test))

if (y_test == 1).sum() == 0:
    print("\n⚠ WARNING: Test set contains 0 positives. Metrics will be unreliable.")
    print("  Consider: using cross-validation evaluation instead of a single hold-out test.")

print("="*100)



CELL 3.3: TRAIN/TEST SPLIT
✓ Split completed using random_state=42
Train shape: (200, 43)  Test shape: (50, 43)
y_train positives: 2 / 200
y_test positives: 1 / 50


---

## CELL 3.4: BUILD PREPROCESSING PIPELINE (NUMERIC + CATEGORICAL)

### What We Do
- Detect numeric and categorical columns.
- Build a `ColumnTransformer` pipeline:
  - Numeric: impute median + StandardScaler
  - Categorical: impute most_frequent + OneHotEncoder

### Why
- Ensures consistent transformation between train and test.
- Prevents leakage (fit only on training data).


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

print("\n" + "="*100)
print("CELL 3.4: PREPROCESSING PIPELINE")
print("="*100)

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("✓ Preprocessing pipeline created")
print("="*100)



CELL 3.4: PREPROCESSING PIPELINE
Numeric features: 3
Categorical features: 40
✓ Preprocessing pipeline created


---

## CELL 3.5: FIT TRANSFORM (TRAIN) + TRANSFORM (TEST)

### What We Do
- Fit preprocessing on training data only.
- Transform both train and test.
- Produce model-ready matrices.


In [6]:
print("\n" + "="*100)
print("CELL 3.5: TRANSFORM DATA")
print("="*100)

X_train_processed = preprocess.fit_transform(X_train)
X_test_processed = preprocess.transform(X_test)

feature_names = preprocess.get_feature_names_out()

print("✓ Transformation complete")
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)
print("Total engineered features:", len(feature_names))

# Convert to DataFrame for easier inspection
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

print("\nSample of processed features:")
print(X_train_df.head(3))

print("="*100)



CELL 3.5: TRANSFORM DATA
✓ Transformation complete
X_train_processed shape: (200, 162)
X_test_processed shape: (50, 162)
Total engineered features: 162

Sample of processed features:
        Age  lab_results.IgM OD_Values   lab_results.IgG OD_Values  \
0 -1.034400                   -0.363022                  -0.217112   
1  1.419999                   -0.273688                  -0.358037   
2 -0.711453                    1.082569                   0.337779   

   Gender_Female  Gender_Male  Occupation_Aluminium  Occupation_Baby  \
0            1.0          0.0                   0.0              0.0   
1            0.0          1.0                   0.0              0.0   
2            0.0          1.0                   0.0              0.0   

   Occupation_Business Man  Occupation_Business Woman  Occupation_Carpentry  \
0                      0.0                        0.0                   0.0   
1                      0.0                        0.0                   0.0   
2        

---

## CELL 3.6: BASELINE MODEL (LOGISTIC REGRESSION) + QUICK METRICS

### Why Baseline?
We need a simple model to ensure the pipeline works end-to-end.

### What We Do
- Train Logistic Regression with `class_weight='balanced'`.
- Evaluate with confusion matrix + classification report.


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

print("\n" + "="*100)
print("CELL 3.6: BASELINE MODEL")
print("="*100)

baseline_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)
baseline_model.fit(X_train_processed, y_train)

y_pred = baseline_model.predict(X_test_processed)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

print("\nNOTE:")
print("- With only 3 positives total, test metrics may fluctuate heavily.")
print("- In Section 4, we should prefer stratified cross-validation.")

print("="*100)



CELL 3.6: BASELINE MODEL

Confusion Matrix:
[[48  1]
 [ 1  0]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9796    0.9796    0.9796        49
           1     0.0000    0.0000    0.0000         1

    accuracy                         0.9600        50
   macro avg     0.4898    0.4898    0.4898        50
weighted avg     0.9600    0.9600    0.9600        50


NOTE:
- With only 3 positives total, test metrics may fluctuate heavily.
- In Section 4, we should prefer stratified cross-validation.


---

## CELL 3.7: SMOKE TEST - SMOTE PIPELINE (CORRECT WAY)

### Important
SMOTE must be applied **only on training data**, and inside a pipeline.

### Warning
You have only **3 positives**, so SMOTE may be unstable. Still included for experimentation.


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import f1_score

print("\n" + "="*100)
print("CELL 3.7: SMOTE PIPELINE (OPTIONAL)")
print("="*100)

smote_pipeline = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=RANDOM_SEED, k_neighbors=1)),
    ("model", LogisticRegression(max_iter=2000, class_weight=None, random_state=RANDOM_SEED))
])

smote_pipeline.fit(X_train, y_train)
y_pred_smote = smote_pipeline.predict(X_test)

print("Confusion Matrix (SMOTE):")
print(confusion_matrix(y_test, y_pred_smote))

print("\nClassification Report (SMOTE):")
print(classification_report(y_test, y_pred_smote, digits=4))

print("\nF1 Score:", f1_score(y_test, y_pred_smote, zero_division=0))
print("="*100)


---

## CELL 3.9: VALIDATION CHECKPOINT (SECTION 3)

### Success Criteria
- `df_clean` exists
- `X_train_processed` and `X_test_processed` created
- Target is binary numeric
- No NaNs in processed matrices
- Ready for hyperparameter tuning & model comparison in next section


In [ ]:
print("\n" + "="*100)
print("CELL 3.9: VALIDATION CHECKPOINT - SECTION 3")
print("="*100)

checks = {
    "df_clean exists": "df_clean" in globals(),
    "X_train exists": "X_train" in globals(),
    "X_train_processed exists": "X_train_processed" in globals(),
    "y is binary": set(pd.Series(y).unique()) == {0, 1},
    "No NaNs in X_train_processed": np.isnan(X_train_processed).sum() == 0,
    "No NaNs in X_test_processed": np.isnan(X_test_processed).sum() == 0
}

for k, v in checks.items():
    print(("✓" if v else "✗"), k)

all_passed = all(checks.values())

print("\n" + "="*100)
if all_passed:
    print("✓✓✓ SECTION 3 COMPLETE - READY FOR MODEL TRAINING/TUNING ✓✓✓")
else:
    print("✗✗✗ SECTION 3 INCOMPLETE - FIX ISSUES ABOVE ✗✗✗")
print("="*100)
